### Instalação de dependências

In [1]:
!pip install -q datasets sentence-transformers spacy scipy scikit-learn
!python -m spacy download pt_core_news_lg -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 568.2/568.2 MB 2.4 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('pt_core_news_lg')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


## Carga do dataset ASSIN 2

In [2]:
import pandas as pd
from datasets import load_dataset

dataset = load_dataset("nilc-nlp/assin2")

df_test = pd.DataFrame(dataset['test'])

print(f"Total de linhas no teste: {len(df_test)}")
df_test[['premise', 'hypothesis', 'relatedness_score']].head()

README.md:   0%|          | 0.00/5.23k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/376k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/157k [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/34.1k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/6500 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2448 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/500 [00:00<?, ? examples/s]

Total de linhas no teste: 2448


,premise,hypothesis,relatedness_score
0,O cachorro caramelo está assistindo um cachorr...,Um cachorro de estimação está de pé no banco e...,3.80
1,O cara está fazendo exercícios no chão,Um cara está fazendo exercícios,3.75
2,Um cachorro grande e um cachorro pequenino est...,Um cachorro grande e um cachorro pequenino est...,4.40
3,Um menino jovem vestindo um traje de banho ver...,Um menino jovem está vestindo um maiô vermelho...,3.80
4,Um cara velho com uma barba que é cinza está a...,Um cara velho com uma barba grisalha está anda...,4.75


## Pré-processamento

In [3]:
import spacy

nlp = spacy.load("pt_core_news_lg")

def extrair_tuplas_sintaticas(texto):
    '''
    Tokeniza, remove pontuação/tokens não-alfabéticos e stopwords,
    e retorna o conjunto de tuplas (lema, rótulo de dependência).
    '''
    doc = nlp(texto.lower())
    tuplas = set(
        (token.lemma_, token.dep_)
        for token in doc
        if token.is_alpha and not token.is_stop
    )
    return tuplas

##Estratégia 1 — Sintática / Sobreposição de tokens anotados com papel sintático

In [4]:
def calcular_overlap_jaccard(premise, hypothesis):
    tuplas_p = extrair_tuplas_sintaticas(premise)
    tuplas_h = extrair_tuplas_sintaticas(hypothesis)

    uniao = tuplas_p.union(tuplas_h)
    if len(uniao) == 0:
        return 1.0

    intersecao = tuplas_p.intersection(tuplas_h)
    jaccard = len(intersecao) / len(uniao)

    score_mapeado = 1.0 + (jaccard * 4.0)
    return score_mapeado

print("Rodando a Estratégia Sintática (tokens lematizados, sem stopwords, normalização Jaccard)...")
df_test['pred_sintatica'] = df_test.apply(
    lambda row: calcular_overlap_jaccard(row['premise'], row['hypothesis']), axis=1
)
print("Concluído!")

Rodando a Estratégia Sintática (tokens lematizados, sem stopwords, normalização Jaccard)...
Concluído!


###Comparação estrutural de fato, usando a árvore de dependência

Aqui, cada token é representado pelo seu **caminho da raiz da árvore até o próprio token** (sequência de rótulos de dependência subindo por `token.head` até a raiz). Comparar caminhos raiz→token é uma forma simples, mas genuinamente estrutural, de capturar posição relativa na árvore — dois tokens só "casam" se tiverem o mesmo lema **e** o mesmo caminho sintático até a raiz, e não apenas o mesmo rótulo de dependência isolado.

In [5]:
def caminho_raiz_token(token):
    caminho = []
    t = token
    while t.head != t:
        caminho.append(t.dep_)
        t = t.head
    caminho.append(t.dep_)
    return tuple(reversed(caminho))

def extrair_tuplas_estruturais(texto):
    doc = nlp(texto.lower())
    return set(
        (token.lemma_, caminho_raiz_token(token))
        for token in doc
        if token.is_alpha and not token.is_stop
    )

def calcular_overlap_estrutural(premise, hypothesis):
    tuplas_p = extrair_tuplas_estruturais(premise)
    tuplas_h = extrair_tuplas_estruturais(hypothesis)

    uniao = tuplas_p.union(tuplas_h)
    if len(uniao) == 0:
        return 1.0

    intersecao = tuplas_p.intersection(tuplas_h)
    jaccard = len(intersecao) / len(uniao)
    return 1.0 + (jaccard * 4.0)

print("Rodando a variante estrutural (bônus, caminho raiz->token)...")
df_test['pred_sintatica_estrutural'] = df_test.apply(
    lambda row: calcular_overlap_estrutural(row['premise'], row['hypothesis']), axis=1
)
print("Concluído!")

Rodando a variante estrutural (bônus, caminho raiz->token)...
Concluído!


In [6]:
import plotly.express as px
import plotly.graph_objects as go

fig_dist = px.histogram(
    df_test,
    x='pred_sintatica_estrutural',
    nbins=30,
    title='Distribuição do Overlap Sintático Estrutural',
    labels={'pred_sintatica_estrutural': 'Score Estrutural (1.0 a 5.0)'},
    template='plotly_white'
)
fig_dist.update_layout(
    yaxis_title='Frequência (Quantidade de Pares)',
    bargap=0.05
)
fig_dist.show()

In [7]:
coluna_label_real = 'entailment_judgment'

if coluna_label_real in df_test.columns:
    fig_box = px.box(
        df_test,
        x=coluna_label_real,
        y='pred_sintatica_estrutural',
        color=coluna_label_real,
        title='Capacidade de Separação do Score Estrutural por Classe Real',
        labels={
            'pred_sintatica_estrutural': 'Score Estrutural',
            coluna_label_real: 'Classe Real'
        },
        template='plotly_white'
    )
    fig_box.show()

##Estratégia 2 - Neural / Transformers (BERTimbau + Sentence Transformers)

In [8]:
from sentence_transformers import SentenceTransformer, util
import numpy as np

model = SentenceTransformer('neuralmind/bert-base-portuguese-cased')

print("Gerando Embeddings e calculando as Similaridades...")
premises = df_test['premise'].tolist()
hypotheses = df_test['hypothesis'].tolist()

embeddings_p = model.encode(premises, convert_to_tensor=True)
embeddings_h = model.encode(hypotheses, convert_to_tensor=True)

cos_scores = util.cos_sim(embeddings_p, embeddings_h)
pred_cosseno = np.diag(cos_scores.cpu().numpy())

# Mapeia o cosseno (que varia de -1 a 1) para a escala de 1 a 5
df_test['pred_neural'] = 1.0 + ((pred_cosseno + 1.0) / 2.0) * 4.0
print("Concluído!")

config.json:   0%|          | 0.00/647 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: neuralmind/bert-base-portuguese-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/43.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/210k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Gerando Embeddings e calculando as Similaridades...
Concluído!


##Métricas: Correlação de Pearson e MSE

In [9]:
from scipy.stats import pearsonr
from sklearn.metrics import mean_squared_error

y_true = df_test['relatedness_score'].values

# Estratégia 1
p_corr_sintatica, _ = pearsonr(y_true, df_test['pred_sintatica'].values)
mse_sintatica = mean_squared_error(y_true, df_test['pred_sintatica'].values)

# Estratégia 1.1
p_corr_estrutural, _ = pearsonr(y_true, df_test['pred_sintatica_estrutural'].values)
mse_estrutural = mean_squared_error(y_true, df_test['pred_sintatica_estrutural'].values)

# Estratégia 2
p_corr_neural, _ = pearsonr(y_true, df_test['pred_neural'].values)
mse_neural = mean_squared_error(y_true, df_test['pred_neural'].values)

print("\n" + "="*60)
print("               TABELA DE RESULTADOS FINAIS               ")
print("="*60)
print(f"{'Estratégia':<38} | {'Pearson (r)':<12} | {'MSE':<6}")
print("-"*60)
print(f"{'1. Sintática (lema+stopwords+Jaccard)':<38} | {p_corr_sintatica:<12.4f} | {mse_sintatica:<6.4f}")
print(f"{'1.1 Sintática estrutural (bônus)':<38} | {p_corr_estrutural:<12.4f} | {mse_estrutural:<6.4f}")
print(f"{'2. Neural (Transformers)':<38} | {p_corr_neural:<12.4f} | {mse_neural:<6.4f}")
print("="*60)
print("Nota: na Correlação de Pearson, quanto mais próximo de 1.0, melhor.")
print("Nota: no MSE, quanto mais próximo de 0.0, melhor.")


               TABELA DE RESULTADOS FINAIS               
Estratégia                             | Pearson (r)  | MSE   
------------------------------------------------------------
1. Sintática (lema+stopwords+Jaccard)  | 0.4584       | 2.7983
1.1 Sintática estrutural (bônus)       | 0.4234       | 3.3768
2. Neural (Transformers)               | 0.6715       | 2.1435
Nota: na Correlação de Pearson, quanto mais próximo de 1.0, melhor.
Nota: no MSE, quanto mais próximo de 0.0, melhor.


###Teste de significância estatística

In [10]:
from scipy.stats import norm

def steiger_z_test(r_gold_A, r_gold_B, r_A_B, n):
    '''
    Teste de Steiger (1980) para comparar duas correlacoes dependentes
    que compartilham uma variavel em comum (aqui, o gabarito).

    r_gold_A : correlacao entre o gabarito e as predicoes do modelo A
    r_gold_B : correlacao entre o gabarito e as predicoes do modelo B
    r_A_B    : correlacao entre as predicoes do modelo A e do modelo B
    n        : tamanho da amostra
    '''
    z_A = np.arctanh(r_gold_A)
    z_B = np.arctanh(r_gold_B)

    rm2 = (r_gold_A**2 + r_gold_B**2) / 2.0
    denom = (1 - rm2)
    f = (1 - r_A_B) / (2 * denom) if denom != 0 else 1.0
    f = min(f, 1.0)
    h = (1 - f * rm2) / denom if denom != 0 else 1.0

    z_stat = (z_A - z_B) * np.sqrt((n - 3) / (2 * (1 - r_A_B) * h))
    p_valor = 2 * (1 - norm.cdf(abs(z_stat)))
    return z_stat, p_valor

import numpy as np

n = len(df_test)
r_sint_neural, _ = pearsonr(df_test['pred_sintatica'].values, df_test['pred_neural'].values)

z_stat, p_valor = steiger_z_test(p_corr_sintatica, p_corr_neural, r_sint_neural, n)

print(f"r(gold, sintática)      = {p_corr_sintatica:.4f}")
print(f"r(gold, neural)         = {p_corr_neural:.4f}")
print(f"r(sintática, neural)    = {r_sint_neural:.4f}")
print(f"\nTeste de Steiger (1980): z = {z_stat:.4f} | p-valor = {p_valor:.6f}")
if p_valor < 0.05:
    print("=> A diferença entre as duas correlações é estatisticamente significativa (p < 0.05).")
else:
    print("=> Não há evidência estatística suficiente para afirmar que as correlações diferem (p >= 0.05).")

r(gold, sintática)      = 0.4584
r(gold, neural)         = 0.6715
r(sintática, neural)    = 0.6778

Teste de Steiger (1980): z = -16.7165 | p-valor = 0.000000
=> A diferença entre as duas correlações é estatisticamente significativa (p < 0.05).


##Visualizações

In [11]:
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import numpy as np

# 1. GRÁFICO DE LINHAS: Tendência e Calibração (Gabarito vs Predições)
df_sample = df_test.sample(50, random_state=42).sort_values(by='relatedness_score')

fig_linhas = go.Figure()
fig_linhas.add_trace(go.Scatter(y=df_sample['relatedness_score'], mode='lines+markers', name='Gabarito Real', line=dict(color='black', width=3)))
fig_linhas.add_trace(go.Scatter(y=df_sample['pred_neural'], mode='lines', name='Estratégia Neural', line=dict(color='blue', dash='dash')))
fig_linhas.add_trace(go.Scatter(y=df_sample['pred_sintatica'], mode='lines', name='Estratégia Sintática (corrigida)', line=dict(color='red', dash='dot')))

fig_linhas.update_layout(
    title='Comparação de Tendência e Alinhamento (Amostra de 50 Pares)',
    xaxis_title='Índice do Par (Ordenado pelo Gabarito)',
    yaxis_title='Relatedness Score (1 a 5)',
    template='plotly_white',
    legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.01)
)
fig_linhas.show()


# 2. GRÁFICO DE DISPERSÃO (Scatter Plot)
fig_scatter = make_subplots(rows=1, cols=2, subplot_titles=("Estratégia Sintática (corrigida)", "Estratégia Neural"))

fig_scatter.add_trace(
    go.Scatter(x=df_test['relatedness_score'], y=df_test['pred_sintatica'], mode='markers',
               marker=dict(color='red', opacity=0.4), name='Sintática'), row=1, col=1
)
fig_scatter.add_trace(
    go.Scatter(x=df_test['relatedness_score'], y=df_test['pred_neural'], mode='markers',
               marker=dict(color='blue', opacity=0.4), name='Neural'), row=1, col=2
)

for col in [1, 2]:
    fig_scatter.add_trace(go.Scatter(x=[1, 5], y=[1, 5], mode='lines', line=dict(color='gray', dash='dash'), showlegend=False), row=1, col=col)

fig_scatter.update_layout(
    title='Dispersão das Predições em Relação ao Gabarito Ideal (Linha Cinza)',
    xaxis_title='Gabarito Real',
    yaxis_title='Predição',
    template='plotly_white',
    showlegend=False
)
fig_scatter.show()


# 3. HISTOGRAMA DE ERROS ABSOLUTOS
df_test['erro_sintatica'] = np.abs(df_test['relatedness_score'] - df_test['pred_sintatica'])
df_test['erro_neural'] = np.abs(df_test['relatedness_score'] - df_test['pred_neural'])

fig_hist = go.Figure()
fig_hist.add_trace(go.Histogram(x=df_test['erro_sintatica'], name='Erro Sintática (corrigida)', marker_color='red', opacity=0.6))
fig_hist.add_trace(go.Histogram(x=df_test['erro_neural'], name='Erro Neural', marker_color='blue', opacity=0.6))

fig_hist.update_layout(
    title='Distribuição do Erro Absoluto',
    xaxis_title='Magnitude do Erro (Valor Absoluto)',
    yaxis_title='Frequência de Ocorrência',
    barmode='overlay',
    template='plotly_white'
)
fig_hist.show()